In [9]:
import os
import json
from dotenv import load_dotenv
from rich.console import Console

from openai import OpenAI
from agents import Agent, Runner, trace, function_tool
from IPython.display import display, HTML

In [10]:
display(HTML("<img src='../../images/agentic-loop-tool-execution.png' width='90%' height='200px' />"))

display(HTML("""
    <h2>Example:- </h2>
    <font style='color:lightgreen'>Step 1:-</font> Ask LLM to propose hard questions to judge IQ. </br></br>
    <font style='color:lightgreen'>Step 2:-</font> Ask 2nd LLM to generated todo list for solving generated question in subsequent prompt.

    <p><font style='color:lightgreen'>Use OpenAI Agent Framework</font> :- </br>
        <li>No need to define meta json. Just use attributes to denote function tools.</li>
        <li>OpenAI Agent platform provides tracing capabilities to track complete execution progress.</li>
    </p>
    """))

In [11]:
# List of TODO and COMPLETED #
todos = []
completed = []

load_dotenv(override=True)

def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)


def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

@function_tool
def create_todos(descriptions: list[str]) -> str:
    """Add new todos from a list of descriptions and return the full list"""
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

@function_tool
def mark_complete(index: int, completion_notes: str) -> str:
    """Mark complete the todo at the given position (starting from 1) and return the full list"""
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

tools = [create_todos, mark_complete]

openai = OpenAI()

question_generator_prompt = "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."
messages = [{"role": "user", "content": question_generator_prompt}]
questonGeneratorResponse = openai.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
    )

question = question = questonGeneratorResponse.choices[0].message.content

print(f"Generated Question: {question} \n\n")

print(tools)

print("\n\n")

todo_planner_executor_system_prompt = """
You are  a todo list planner and executor. For a given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
todoPlannerAndExecutorAgent = Agent(
    name="Todo List Generator and Executor",
    instructions= todo_planner_executor_system_prompt,
    tools= tools
)

print("Trace Available at: https://platform.openai.com/logs?api=traces")

result = ""

with trace("TO List Planner and Executor Trace"):
    result = await Runner.run(todoPlannerAndExecutorAgent, question)

print(f"\n\n {result}")

Generated Question: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? 


[FunctionTool(name='create_todos', description='Add new todos from a list of descriptions and return the full list', params_json_schema={'properties': {'descriptions': {'items': {'type': 'string'}, 'title': 'Descriptions', 'type': 'array'}}, 'required': ['descriptions'], 'title': 'create_todos_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001494C4991C0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='mark_complete', description='Mark complete the todo at the given position (starting from 1) and return the full list', params_json_schema={'properties': {'index': {'title': 'Index', 'type': 'integer'}, 'completion_notes': {'title': 'Completion Notes', 'type': 'string'}}, 'require

Todo #1: Let the cost of the ball be x dollars.
Todo #2: Express the cost of the bat in terms of x (bat = x + $1.00).
Todo #3: Write and solve the equation: x + (x + 1.00) = 1.10.
Todo #4: Calculate the value of x.

Let x = cost of the ball in dollars.

Todo #1: Let the cost of the ball be x dollars.
Todo #2: Express the cost of the bat in terms of x (bat = x + $1.00).
Todo #3: Write and solve the equation: x + (x + 1.00) = 1.10.
Todo #4: Calculate the value of x.

The cost of the bat = x + $1.00.

...

Equation: x + (x + 1.00) = 1.10.

Todo #1: Let the cost of the ball be x dollars.
Todo #2: Express the cost of the bat in terms of x (bat = x + $1.00).
Todo #3: Write and solve the equation: x + (x + 1.00) = 1.10.
Todo #4: Calculate the value of x.

x + x + 1.00 = 1.10 → 2x + 1.00 = 1.10 → 2x = 0.10 → x = 0.05. The ball costs $0.05.

Todo #1: Let the cost of the ball be x dollars.
Todo #2: Express the cost of the bat in terms of x (bat = x + $1.00).
Todo #3: Write and solve the equation: x + (x + 1.00) = 1.10.
Todo #4: Calculate the value of x.



 RunResult:
- Last agent: Agent(name="Todo List Generator and Executor", ...)
- Final output (str):
    🟩 Solution Steps:
    1. Let x = cost of the ball in dollars.
    2. The bat costs x + $1.00.
    3. Equation: x + (x + 1.00) = 1.10.
    4. Solving: 2x + 1.00 = 1.10 → 2x = 0.10 → x = 0.05.
    
    ⭐ The ball costs **$0.05**.
- 11 new item(s)
- 6 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)
